# 08 · End-to-end RAG evaluation
Runs the final pipeline from `rag.py` (hybrid search + reranker + local LLM) on the eval set
and measures the answers.

In [1]:
# ===== CELL 1: Load the pipeline from rag.py =====

import sys                                              # sys: lets us tell Python where to find rag.py
sys.path.append("..")                                   # rag.py is in the project root, one folder up

from rag import RagPipeline, NOT_FOUND                  # The pipeline class and the refusal sentence

rag = RagPipeline(provider="groq")                      # This run uses the hosted model; use provider="ollama" for the local one
print("Provider:", rag.provider, "| Model:", rag.model) # Confirm the settings

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Provider: groq | Model: openai/gpt-oss-20b


In [ ]:
# ===== CELL 2: Quick test =====
# The question that caused the AddTransient hallucination, plus two that retrieval used to miss.

def show(result):                                       # Print an answer and its sources
    print("Q:", result["question"], "\n")
    print(result["answer"], "\n")
    for i, h in enumerate(result["hits"], start=1):
        print(f"  [{i}] {h['where']}  (rerank {h['rerank_score']})")
    print(f"\n({result['total_seconds']} s total, {result['llm_seconds']} s in the LLM)")
    print("=" * 100)

show(rag.ask("How do I register a service so a new instance is created per HTTP request?"))   # F02: still a retrieval miss, so does the LLM refuse or hallucinate?
show(rag.ask("Which URL does Kestrel listen on if no endpoints are configured?"))             # F15: the reranker moved this to #1
show(rag.ask("How do I configure a custom error page for the Production environment?"))       # F11: hybrid + reranker moved this to #1

In [2]:
# ===== CELL 3: Run all 50 questions through the full pipeline =====
# Each answer is saved to a file immediately, so if this stops, rerunning it resumes where it left off.

import json                                                   # json: write each result as one line of JSON
import pandas as pd                                           # pandas: tables
from pathlib import Path                                      # Path: file paths

evals = pd.read_json("../evals/eval_set.jsonl", lines=True)   # All 50 questions (including the 5 out-of-scope ones)

RUN_NAME = f"hybrid_rerank__{rag.model.replace('/', '-')}"    # e.g. "hybrid_rerank__openai-gpt-oss-20b" ("/" isn't allowed in file names)
OUT = Path(f"../reports/answers__{RUN_NAME}.jsonl")           # Where the answers are saved

done = set()                                                  # IDs already answered in an earlier run
if OUT.exists():                                              # If the file exists from before...
    done = {json.loads(line)["id"] for line in OUT.open(encoding="utf-8")}   # ...read which question IDs it already contains
print(f"Already done: {len(done)}/{len(evals)}")

with OUT.open("a", encoding="utf-8") as f:              # Open the file in "append" mode: new lines are added at the end
    for q in evals.itertuples():                        # For each question...
        if q.id in done:                                # ...skip it if it was answered before
            continue
        result = rag.ask(q.question)                    # Run the full pipeline: retrieve -> rerank -> LLM
        result["id"] = q.id                             # Remember which question this was
        f.write(json.dumps(result, ensure_ascii=False) + "\n")   # Save it as one line
        f.flush()                                       # Write to disk now (not later), so nothing is lost if it stops
        print(f"{q.id} done in {result['total_seconds']} s")    # Progress message

print("All questions answered!")

Already done: 50/50
All questions answered!


In [8]:
# ===== CELL 4: Automatic answer metrics =====
# Measures what we can check without reading the answers: retrieval hits, refusals, citations and speed.

import re                                               # re: text patterns

def norm(text):                                         # Collapse whitespace and lowercase, for comparing text
    return re.sub(r"\s+", " ", text).lower()

answers = pd.read_json(OUT, lines=True).merge(evals.drop(columns="question"), on="id")   # Answers joined with their expected article and evidence

def retrieval_hit(row):                                 # Did any of the 5 chunks given to the LLM contain the evidence?
    if row["doc_id"] is None or pd.isna(row["doc_id"]):  # Out-of-scope questions have no correct chunk...
        return None                                     # ...so this check doesn't apply
    return any(
        h["chunk_id"].split("::")[0] == row["doc_id"]   # The chunk is from the expected article...
        and norm(row["evidence"]) in norm(h["text"])    # ...and contains the evidence phrase
        for h in row["hits"]
    )

answers["retrieval_hit"] = answers.apply(retrieval_hit, axis=1)    # Apply the check to every row
answers["refused"] = answers["answer"].str.replace("’", "'").str.lower().str.contains("couldn't find this in the documentation")   # Did it give the refusal sentence? (also handles curly apostrophes)
answers["cited"] = answers["answer"].str.contains(r"\[\d+\]")      # Does the answer contain a citation like [1]?
answers["failed"] = answers["answer"].str.startswith("[Generation failed")   # Did the LLM call fail?

ins = answers[answers["type"] != "out_of_scope"]        # The 45 in-scope questions
oos = answers[answers["type"] == "out_of_scope"]        # The 5 out-of-scope questions

print(f"Generation failures:            {answers['failed'].sum()}")
print(f"Retrieval Hit@5 (in-scope):     {ins['retrieval_hit'].mean():.0%}")
print(f"Answered (in-scope, not refused): {(~ins['refused']).mean():.0%}")
print(f"Citation rate (answered only):  {ins.loc[~ins['refused'], 'cited'].mean():.0%}")
print(f"Correct refusals (out-of-scope): {oos['refused'].sum()}/{len(oos)}")
print(f"Average time per question:      {answers['total_seconds'].mean():.0f} s\n")

# The key safety table: what did the model do when retrieval missed?
print("In-scope questions: retrieval result vs. model behaviour")
display(pd.crosstab(
    ins["retrieval_hit"].map({True: "retrieval hit", False: "retrieval MISS"}),
    ins["refused"].map({True: "refused", False: "answered"}),
))

Generation failures:            0
Retrieval Hit@5 (in-scope):     78%
Answered (in-scope, not refused): 98%
Citation rate (answered only):  75%
Correct refusals (out-of-scope): 5/5
Average time per question:      8 s

In-scope questions: retrieval result vs. model behaviour


refused,answered,refused
retrieval_hit,,
retrieval MISS,9,1
retrieval hit,35,0


In [ ]:
# ===== CELL 5: Export a grading sheet =====
# Creates an Excel file with one row per question: the question, expected answer, model answer
# and sources, plus dropdown columns for your grades. It never overwrites a sheet you've started grading.

from openpyxl import load_workbook                      # load_workbook: open an Excel file to format it
from openpyxl.styles import Alignment, Font, PatternFill   # Styles: text wrapping, bold headers, header colour
from openpyxl.worksheet.datavalidation import DataValidation   # DataValidation: dropdown lists in cells

GRADING = Path(f"../evals/grading__{RUN_NAME}.xlsx")    # Where the grading sheet goes

def hit_label(v):                                       # Turn True/False/None into readable text
    if v is None or pd.isna(v):
        return "n/a"                                    # Out-of-scope questions
    return "hit" if v else "MISS"

sheet = pd.DataFrame({                                  # Build the sheet's columns
    "id": answers["id"],
    "area": answers["area"],
    "type": answers["type"],
    "retrieval": answers["retrieval_hit"].map(hit_label),   # Did the right chunk reach the LLM?
    "question": answers["question"],
    "expected_answer": answers["expected_answer"],
    "model_answer": answers["answer"],
    "sources": answers["hits"].map(lambda hs: "\n".join(f"[{i}] {h['where']}" for i, h in enumerate(hs, start=1))),   # The 5 sources, one per line
    "correct": "",                                      # You fill this: yes / partial / no
    "hallucinated": "",                                 # You fill this: yes / no
    "notes": "",                                        # Optional comments
})

if GRADING.exists():                                    # Safety: never overwrite grades you've already entered
    print(f"{GRADING.name} already exists, so it was NOT overwritten. Delete it first if you really want a fresh sheet.")
else:
    sheet.to_excel(GRADING, index=False)                # Write the sheet
    wb = load_workbook(GRADING)                         # Reopen it to make it readable
    ws = wb.active                                      # The first (only) worksheet
    widths = {"A": 6, "B": 13, "C": 12, "D": 10, "E": 40, "F": 45, "G": 70, "H": 45, "I": 10, "J": 13, "K": 30}   # Column widths
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows(min_row=2):                 # Every data row...
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")   # ...wraps long text and aligns to the top
    for cell in ws[1]:                                  # The header row...
        cell.font = Font(bold=True)                     # ...in bold...
        cell.fill = PatternFill("solid", fgColor="DDEBF7")   # ...with a light blue background
    ws.freeze_panes = "E2"                              # Keep the header row and the id/area/type/retrieval columns visible while scrolling
    correct_dv = DataValidation(type="list", formula1='"yes,partial,no"', allow_blank=True)   # Dropdown for "correct"
    halluc_dv = DataValidation(type="list", formula1='"yes,no"', allow_blank=True)            # Dropdown for "hallucinated"
    ws.add_data_validation(correct_dv)
    ws.add_data_validation(halluc_dv)
    correct_dv.add(f"I2:I{ws.max_row}")                 # Apply the dropdowns to every data row
    halluc_dv.add(f"J2:J{ws.max_row}")
    wb.save(GRADING)                                    # Save
    print("Grading sheet created:", GRADING.resolve())  # Show the full path

In [9]:
# ===== CELL 6: Results from your grades =====
# Reads your graded sheet and computes correctness and hallucination rates.

g = pd.read_excel(GRADING, keep_default_na=False)       # Load your grades; keep_default_na=False stops pandas turning the text "n/a" into an empty value                             # Load your grades
g["correct"] = g["correct"].astype(str).str.strip().str.lower()        # Tidy the values (remove spaces, lowercase)
g["hallucinated"] = g["hallucinated"].astype(str).str.strip().str.lower()

ungraded = (~g["correct"].isin(["yes", "partial", "no"])).sum()        # Rows without a valid grade
if ungraded:
    print(f"⚠️ {ungraded} rows are not graded yet. Results below are incomplete.\n")

g["score"] = g["correct"].map({"yes": 1.0, "partial": 0.5, "no": 0.0})  # Points: yes = 1, partial = 0.5, no = 0
g["halluc"] = g["hallucinated"] == "yes"                                # True/False

def grade_summary(df):                                  # One row of results for a group of questions
    return pd.Series({
        "n": len(df),
        "Correct (yes)": (df["correct"] == "yes").mean(),
        "Partial": (df["correct"] == "partial").mean(),
        "Wrong (no)": (df["correct"] == "no").mean(),
        "Score": df["score"].mean(),                    # Average points (partial counts as half)
        "Hallucinated": df["halluc"].mean(),
    })

fmt = {"n": "{:.0f}", "Correct (yes)": "{:.0%}", "Partial": "{:.0%}", "Wrong (no)": "{:.0%}", "Score": "{:.0%}", "Hallucinated": "{:.0%}"}
overall = grade_summary(g).to_frame(RUN_NAME).T          # Overall results
display(overall.style.format(fmt))
display(g.groupby("retrieval").apply(grade_summary).style.format(fmt))   # Split by retrieval hit / MISS / n/a
display(g.groupby("area").apply(grade_summary).style.format(fmt))        # Split by area
display(g.groupby("type").apply(grade_summary).style.format(fmt))        # Split by question type

summary_path = Path(f"../reports/generation_summary__{RUN_NAME}.csv")   # One summary file per run, named after the run, so runs don't overwrite each other
overall.to_csv(summary_path)                                           # Save it
print("Saved:", summary_path)

,n,Correct (yes),Partial,Wrong (no),Score,Hallucinated
hybrid_rerank__openai-gpt-oss-20b,50,92%,4%,4%,94%,4%


,n,Correct (yes),Partial,Wrong (no),Score,Hallucinated
retrieval,,,,,,
MISS,10,60%,20%,20%,70%,20%
hit,35,100%,0%,0%,100%,0%
n/a,5,100%,0%,0%,100%,0%


,n,Correct (yes),Partial,Wrong (no),Score,Hallucinated
area,,,,,,
blazor,15,100%,0%,0%,100%,0%
fundamentals,15,87%,0%,13%,87%,13%
mvc,15,87%,13%,0%,93%,0%
out_of_scope,5,100%,0%,0%,100%,0%


,n,Correct (yes),Partial,Wrong (no),Score,Hallucinated
type,,,,,,
direct,30,93%,3%,3%,95%,3%
out_of_scope,5,100%,0%,0%,100%,0%
paraphrase,15,87%,7%,7%,90%,7%


Saved: ..\reports\generation_summary__hybrid_rerank__openai-gpt-oss-20b.csv


In [ ]:
# ===== CHECK: compare the two grading sheets =====
old = pd.read_excel("../evals/grading__hybrid_rerank__llama3.2-3b.xlsx", keep_default_na=False)   # The Ollama run's sheet
new = pd.read_excel(GRADING, keep_default_na=False)                                              # The Groq run's sheet

print("New sheet:", GRADING.name)                                                                # Confirm which file is the "new" one
print("Model answers identical:", (old["model_answer"] == new["model_answer"]).sum(), "of", len(new))   # Should be close to 0: different models write different answers
print("Grades identical:", ((old["correct"] == new["correct"]) & (old["hallucinated"] == new["hallucinated"])).sum(), "of", len(new))